Read API Structure
DataFrameReader.format(...).oprions("key", "value").schema(...),load()

Write API Structure
DataFrameWriter.format(...).options("key", "value").mode(...).partitionBy(...).bucketBy(...).sortBy(...).save()

In [8]:

from pyspark.sql import SparkSession

DEFAULT_SPARK_REMOTE = "sc://localhost:15002"
spark = SparkSession.builder.remote(DEFAULT_SPARK_REMOTE).getOrCreate()

spark.version

'4.1.2'

In [9]:
northwind_folder = "s3a://lakehouse/external-locations/sample-data/northwind/"

spark.sql("show catalogs").show()
spark.sql("show schemas").show()

+-------------+
|      catalog|
+-------------+
|spark_catalog|
+-------------+

+------------+
|   namespace|
+------------+
|     default|
|northwind_db|
+------------+



In [10]:
spark.sql("create schema if not exists northwind_db")

spark.sql("use spark_catalog")
spark.sql("use schema northwind_db")

DataFrame[]

In [11]:
from pyspark.sql.functions import col, to_date, to_timestamp

spark.sql("use schema northwind_db")

df = (spark.read
      .format("csv")
      .option("header", "true")
      .load(f"{northwind_folder}Orders.csv")
      .withColumn("OrderDate", to_date(to_timestamp(col("OrderDate")), "format='yyyy-MM-dd'"))
      .withColumn("RequiredDate", to_date(to_timestamp(col("RequiredDate")), "format='yyyy-MM-dd'"))
      #.withColumn("ShippedDate", to_date(col("ShippedDate").cast("timestamp"), "format='yyyy-MM-dd'"))
      #.withColumn("ShipVia", col("ShipVia").cast("int"))
      .withColumn("Freight", col("Freight").cast("decimal(10,2)"))
      )

df.printSchema()

(df.write
 .format("delta")
 .option("overwriteSchema", "true")
 .mode("overwrite")
 .partitionBy("CustomerID")
 .saveAsTable("orders")
 )
spark.sql("show tables").show()



root
 |-- OrderID: string (nullable = true)
 |-- CustomerID: string (nullable = true)
 |-- EmployeeID: string (nullable = true)
 |-- OrderDate: date (nullable = true)
 |-- RequiredDate: date (nullable = true)
 |-- ShippedDate: string (nullable = true)
 |-- ShipVia: string (nullable = true)
 |-- Freight: decimal(10,2) (nullable = true)
 |-- ShipName: string (nullable = true)
 |-- ShipAddress: string (nullable = true)
 |-- ShipCity: string (nullable = true)
 |-- ShipRegion: string (nullable = true)
 |-- ShipPostalCode: string (nullable = true)
 |-- ShipCountry: string (nullable = true)

+------------+-------------+-----------+
|   namespace|    tableName|isTemporary|
+------------+-------------+-----------+
|northwind_db|    employees|      false|
|northwind_db|order_details|      false|
|northwind_db|       orders|      false|
+------------+-------------+-----------+



In [12]:


spark.sql("select * from orders").limit(3).show()

+-------+----------+----------+----------+------------+--------------------+-------+-------+--------------+--------------------+--------+-------------+--------------+-----------+
|OrderID|CustomerID|EmployeeID| OrderDate|RequiredDate|         ShippedDate|ShipVia|Freight|      ShipName|         ShipAddress|ShipCity|   ShipRegion|ShipPostalCode|ShipCountry|
+-------+----------+----------+----------+------------+--------------------+-------+-------+--------------+--------------------+--------+-------------+--------------+-----------+
|  10315|     ISLAT|         4|1996-09-26|  1996-10-24|1996-10-03 00:00:...|      2|  41.76|Island Trading|Garden House Crow...|   Cowes|Isle of Wight|      PO31 7PJ|         UK|
|  10318|     ISLAT|         8|1996-10-01|  1996-10-29|1996-10-04 00:00:...|      2|   4.73|Island Trading|Garden House Crow...|   Cowes|Isle of Wight|      PO31 7PJ|         UK|
|  10321|     ISLAT|         3|1996-10-03|  1996-10-31|1996-10-11 00:00:...|      2|   3.43|Island Tradin

In [13]:

df = (spark.read.csv(f"{northwind_folder}OrdersDetails.csv", header=True)
 .withColumn("UnitPrice", col("UnitPrice").cast("decimal(10,2)"))
 .withColumn("Quantity", col("Quantity").cast("int"))
 .withColumn("Discount", col("Discount").cast("decimal(10,2)"))
 )

df.printSchema()

(df.write
 .format("delta")
 .option("overwriteSchema", "true")
 .mode("overwrite")
 .partitionBy("ProductID")
 .saveAsTable("order_details"))

spark.sql("show tables").show()
spark.sql("select * from order_details").limit(3).show()

root
 |-- OrderID: string (nullable = true)
 |-- ProductID: string (nullable = true)
 |-- UnitPrice: decimal(10,2) (nullable = true)
 |-- Quantity: integer (nullable = true)
 |-- Discount: decimal(10,2) (nullable = true)

+------------+-------------+-----------+
|   namespace|    tableName|isTemporary|
+------------+-------------+-----------+
|northwind_db|    employees|      false|
|northwind_db|order_details|      false|
|northwind_db|       orders|      false|
+------------+-------------+-----------+

+-------+---------+---------+--------+--------+
|OrderID|ProductID|UnitPrice|Quantity|Discount|
+-------+---------+---------+--------+--------+
|  10283|       15|    12.40|      20|    0.00|
|  10323|       15|    12.40|       5|    0.00|
|  10500|       15|    15.50|      12|    0.05|
+-------+---------+---------+--------+--------+



In [14]:
(spark.read
     .csv(f"{northwind_folder}Employees.csv", header=True)
     .write.format("delta")
     .mode("overwrite")
     .option("overwriteSchema", "true")
     .saveAsTable("Employees"))